# Import Library & Load Dataset

In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('customer_support_tickets.csv')

print(f"Dataset Shape: {df.shape}")
df.head(3)

Dataset Shape: (8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0


# Data Cleaning & Feature Engineering

In [2]:
# 1. Cleaning Placeholders in Ticket Description
df['Ticket Description Cleaned'] = df.apply(
    lambda x: str(x['Ticket Description']).replace('{product_purchased}', str(x['Product Purchased'])), 
    axis=1
)

# 2. Convert Date & Time Columns to Datetime Objects
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])
df['First Response Time'] = pd.to_datetime(df['First Response Time'])
df['Time to Resolution'] = pd.to_datetime(df['Time to Resolution'])

# 3. Calculate Response & Resolution Durations (in Hours)
# Response Time Duration (First Response Time - Date of Purchase)
df['First Response Duration (Days)'] = (df['First Response Time'] - df['Date of Purchase']).dt.total_seconds() / (24 * 3600)

# Resolution Duration (Time to Resolution - First Response Time)
df['Resolution Duration (Hours)'] = (df['Time to Resolution'] - df['First Response Time']).dt.total_seconds() / 3600

# Handling negative values in Resolution Duration (Inconsistency Handling)
# Option: Replace negative durations with Absolute Value or NaN to preserve logical consistency
df['Resolution Duration (Hours) Clean'] = df['Resolution Duration (Hours)'].apply(lambda x: abs(x) if pd.notnull(x) else np.nan)

# 4. Fill Missing Values for Export
# Standardize CSAT representation for closed vs open tickets
df['CSAT_Category'] = df['Customer Satisfaction Rating'].map({
    1.0: '1 - Poor',
    2.0: '2 - Fair',
    3.0: '3 - Average',
    4.0: '4 - Good',
    5.0: '5 - Excellent'
}).fillna('Not Rated / Unresolved')

print("Data Cleaning & Feature Engineering Complete!")

Data Cleaning & Feature Engineering Complete!


# EDA

In [3]:
# Check Summary Metrics
print("--- Ticket Status Distribution ---")
print(df['Ticket Status'].value_counts())

print("\n--- Average Resolution Duration (Hours) per Priority ---")
print(df.groupby('Ticket Priority')['Resolution Duration (Hours) Clean'].mean())

print("\n--- Average CSAT per Ticket Channel ---")
print(df.groupby('Ticket Channel')['Customer Satisfaction Rating'].mean())

--- Ticket Status Distribution ---
Ticket Status
Pending Customer Response    2881
Open                         2819
Closed                       2769
Name: count, dtype: int64

--- Average Resolution Duration (Hours) per Priority ---
Ticket Priority
Critical    7.564279
High        8.200922
Low         7.858359
Medium      7.355211
Name: Resolution Duration (Hours) Clean, dtype: float64

--- Average CSAT per Ticket Channel ---
Ticket Channel
Chat            3.083086
Email           2.963889
Phone           2.952243
Social media    2.969298
Name: Customer Satisfaction Rating, dtype: float64


# Export Cleaned Dataset for MySQL Ingestion

In [4]:
# Drop or rearrange columns for final export
clean_df = df.copy()

# Save cleaned file to CSV
clean_df.to_csv('customer_support_tickets_clean.csv', index=False)
print("File 'customer_support_tickets_clean.csv' successfully generated!")

File 'customer_support_tickets_clean.csv' successfully generated!
